In [ ]:
# ============================================================================
# STANDALONE: CORRECT-TRACE DIVERSITY TABLE
# Reports:
#   diversity = 1 - mean pairwise cosine similarity
#   std diversity
#   avg number of correct traces
#   std number of correct traces
# ============================================================================

import os, json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# ----------------------------
# CHANGE THESE IF NEEDED
# ----------------------------
RUN_DIR = "/u/amp20/cvh/LEGAL_REASONING_ENSEMBLE_PROJ/FINAL_RESULTS_FOR_PAPER/five_csv_correct_similarity_run"
OUTPUT_DIR = "/u/amp20/cvh/LEGAL_REASONING_ENSEMBLE_PROJ/FINAL_RESULTS_FOR_PAPER/five_csv_correct_similarity_run/outputs"   # <-- change this

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_BATCH_SIZE = 128

RESULTS_DIR = os.path.join(RUN_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

analysis_targets = [
    ("qwen_only_8", os.path.join(OUTPUT_DIR, "qwen_only_8.csv")),
    ("gemma_only_8", os.path.join(OUTPUT_DIR, "gemma_only_8.csv")),
    ("llama_only_8", os.path.join(OUTPUT_DIR, "llama_only_8.csv")),
    ("mistral_only_8", os.path.join(OUTPUT_DIR, "mistral_only_8.csv")),
    ("mixed_2_each_8", os.path.join(OUTPUT_DIR, "mixed_2_each_8.csv")),
]

SUMMARY_CSV = os.path.join(RESULTS_DIR, "correct_trace_diversity_summary.csv")
PER_QUESTION_CSV = os.path.join(RESULTS_DIR, "correct_trace_diversity_per_question.csv")
SUMMARY_JSON = os.path.join(RESULTS_DIR, "correct_trace_diversity_summary.json")


def norm_answer(x, dataset):
    x = "" if pd.isna(x) else str(x).strip().upper()
    dataset = "" if pd.isna(dataset) else str(dataset).strip().lower()

    if dataset == "sara":
        if x.startswith("ENTAIL"):
            return "ENTAILED"
        if x.startswith("CONTRADICT"):
            return "CONTRADICTED"

    if dataset == "argkp":
        if x.startswith("SUPP") or x.startswith("PRO"):
            return "SUPPORTS"
        if x.startswith("OPP") or x.startswith("CON") or x.startswith("AGAINST"):
            return "OPPOSES"

    if dataset == "gpqa":
        if x and x[0] in "ABCD":
            return x[0]

    return x


def mean_pairwise_cosine_similarity(emb):
    if emb.shape[0] < 2:
        return np.nan

    emb = emb / np.maximum(np.linalg.norm(emb, axis=1, keepdims=True), 1e-12)
    sims = emb @ emb.T
    iu = np.triu_indices(emb.shape[0], k=1)

    return float(np.mean(sims[iu]))


def analyze_csv(csv_path, condition, embedder):
    df = pd.read_csv(csv_path)

    required_cols = ["predicted_label", "gold_label", "dataset", "reasoning", "question_id"]
    for col in required_cols:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].fillna("").astype(str)

    df["pred_norm"] = df.apply(lambda r: norm_answer(r["predicted_label"], r["dataset"]), axis=1)
    df["gold_norm"] = df.apply(lambda r: norm_answer(r["gold_label"], r["dataset"]), axis=1)
    df["is_correct"] = df["pred_norm"] == df["gold_norm"]

    per_q_rows = []

    for (dataset, qid), g in df.groupby(["dataset", "question_id"]):
        correct = g[
            (g["is_correct"]) &
            (g["reasoning"].astype(str).str.strip().str.len() > 0)
        ].copy()

        n_correct = len(correct)
        sim = np.nan
        diversity = np.nan

        if n_correct >= 2:
            texts = correct["reasoning"].astype(str).tolist()
            emb = embedder.encode(
                texts,
                batch_size=EMBED_BATCH_SIZE,
                convert_to_numpy=True,
                normalize_embeddings=False,
                show_progress_bar=False,
            )
            sim = mean_pairwise_cosine_similarity(emb)
            diversity = 1.0 - sim

        per_q_rows.append({
            "condition": condition,
            "dataset": dataset,
            "question_id": qid,
            "n_correct_traces": n_correct,
            "similarity": sim,
            "diversity": diversity,
        })

    qdf = pd.DataFrame(per_q_rows)

    div_vals = qdf["diversity"].dropna().to_numpy()
    correct_vals = qdf["n_correct_traces"].to_numpy()

    summary = {
        "condition": condition,
        "n_questions": int(len(qdf)),
        "avg_diversity": float(np.mean(div_vals)) if len(div_vals) else np.nan,
        "std_diversity": float(np.std(div_vals, ddof=0)) if len(div_vals) else np.nan,
        "avg_correct_traces": float(np.mean(correct_vals)) if len(correct_vals) else np.nan,
        "std_correct_traces": float(np.std(correct_vals, ddof=0)) if len(correct_vals) else np.nan,
    }

    return summary, qdf


print("[loading embedder]")
embedder = SentenceTransformer(EMBED_MODEL)

summary_rows = []
per_question_frames = []

for condition, csv_path in analysis_targets:
    if not os.path.exists(csv_path):
        print(f"[skip missing] {condition}: {csv_path}")
        continue

    print(f"[analyzing] {condition}: {csv_path}")
    s, qdf = analyze_csv(csv_path, condition, embedder)

    summary_rows.append(s)
    per_question_frames.append(qdf)

summary_df = pd.DataFrame(summary_rows)

if summary_df.empty:
    raise RuntimeError(
        "No CSVs were found. Check OUTPUT_DIR and file names: "
        "qwen_only_8.csv, gemma_only_8.csv, llama_only_8.csv, "
        "mistral_only_8.csv, mixed_2_each_8.csv"
    )

if per_question_frames:
    per_question_df = pd.concat(per_question_frames, ignore_index=True)
    per_question_df.to_csv(PER_QUESTION_CSV, index=False)

summary_df.to_csv(SUMMARY_CSV, index=False)

with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary_rows, f, indent=2)

print("\n" + "=" * 90)
print("CORRECT-TRACE DIVERSITY TABLE")
print("=" * 90)

print(summary_df[[
    "condition",
    "avg_diversity",
    "std_diversity",
    "avg_correct_traces",
    "std_correct_traces",
]].to_string(index=False))

print("\n[done]")
print(f"summary CSV      : {SUMMARY_CSV}")
print(f"summary JSON     : {SUMMARY_JSON}")
print(f"per-question CSV : {PER_QUESTION_CSV}")